# Estudo de Extração de Features

Demonstra o **front-end real** usado pelo sistema na inferência
(`app.domain.services.detection.audio_preprocessing.prepare_audio_for_model`,
baseado em `tf.signal`, in-graph):

- **LFCC** (default desde a melhoria P0 — supera o mel em anti-spoofing);
- **log-mel**;
- **raw-audio** (forma de onda PCM 1D).

Configuração: `sample_rate=16 kHz`, `n_fft=512`, `hop_length=128`,
`n_mels = n_lfcc = 80`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

## 1. Gerar um sinal sintético de 1 segundo

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
t = np.linspace(0, 1.0, 16000, endpoint=False)
# Soma de senóides + ruído leve (proxy de voz para demonstração).
wav = (0.6 * np.sin(2 * np.pi * 220 * t)
       + 0.3 * np.sin(2 * np.pi * 440 * t)
       + 0.05 * rng.standard_normal(16000)).astype("float32")
print("waveform:", wav.shape, wav.dtype)

## 2. Extrair os três front-ends reais

In [ ]:
from app.domain.services.detection.audio_preprocessing import (
    prepare_audio_for_model,
)

lfcc = prepare_audio_for_model(
    wav, input_type="spectrogram", input_shape=(100, 80),
    sample_rate=16000, feature_frontend="lfcc",
)
logmel = prepare_audio_for_model(
    wav, input_type="spectrogram", input_shape=(100, 80),
    sample_rate=16000, feature_frontend="logmel",
)
raw = prepare_audio_for_model(
    wav, input_type="raw_audio", input_shape=(16000, 1),
    sample_rate=16000,
)
import numpy as np
print("LFCC   :", np.asarray(lfcc).shape)
print("log-mel:", np.asarray(logmel).shape)
print("raw    :", np.asarray(raw).shape)

## 3. Visualizar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(wav[:1000], color="#3b82f6", lw=0.8)
ax[0].set_title("Forma de onda (1000 amostras)")
ax[1].imshow(np.asarray(lfcc).T, aspect="auto", origin="lower", cmap="magma")
ax[1].set_title("LFCC (80 × T)")
ax[2].imshow(np.asarray(logmel).T, aspect="auto", origin="lower", cmap="magma")
ax[2].set_title("log-mel (80 × T)")
fig.tight_layout()
plt.show()

## Leituras

- `docs/04_FEATURES.md` — features e extratores.
- `docs/09_INFERENCIA.md` — como o `input_contract` garante paridade
  treino↔inferência (o mesmo front-end usado aqui é reproduzido na
  detecção).